In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 8. Local Blog-to-Thread Converter

## What We're Building
A content repurposing tool that takes a blog post and converts it into:
1. **Twitter/X thread** (10-15 tweets)
2. **LinkedIn post** (professional summary)
3. **Email newsletter blurb** (concise version)

**You will learn:**
- Single input → multiple output formats
- Format-specific constraints (character limits, tone)
- Chain composition for content pipelines

**Stack:** Ollama, LangChain

In [ ]:
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:8b", temperature=0.5)
print("LLM ready!")

In [ ]:
# Sample blog post
blog_post = """
# Why Every Developer Should Learn SQL in 2024

SQL isn't sexy. It isn't trending on Twitter. Nobody is making TikToks about JOIN operations.
But here's the uncomfortable truth: SQL remains the single most valuable technical skill
you can learn, regardless of your specialization.

## The Data Behind SQL's Dominance

According to the Stack Overflow Developer Survey 2023, SQL is used by 52% of all developers,
making it the third most popular language after JavaScript and HTML/CSS. More importantly,
it's the #1 skill correlated with higher salaries among developers with 5+ years experience.

Why? Because every application eventually needs to talk to a database. Whether you're
building a web app, training ML models, or analyzing business data, SQL is the interface.

## What Makes SQL Uniquely Valuable

First, SQL is declarative. You describe WHAT you want, not HOW to get it. This makes it
accessible even to non-programmers. A marketing analyst can learn enough SQL in a weekend
to pull their own reports.

Second, SQL is universal. PostgreSQL, MySQL, SQLite, BigQuery, Snowflake — they all speak
SQL. Learn it once, apply it everywhere.

Third, SQL is composable. Complex queries are built from simple building blocks: SELECT,
WHERE, JOIN, GROUP BY. Each concept layers on the previous one.

## The Career Impact

I've seen three patterns in my 10 years of mentoring developers:
1. Developers who know SQL get promoted faster because they can own data decisions
2. Data scientists who know SQL spend 60% less time on data wrangling
3. Product managers who learn SQL become dramatically more effective

## Getting Started

Don't overthink it. Install PostgreSQL locally, load a sample dataset (try the Northwind
database), and start with SELECT statements. Within a week, you'll be writing JOINs.
Within a month, you'll be writing window functions. Within three months, you'll wonder
how you ever worked without it.

The best investment in your career isn't the next JavaScript framework. It's 20 hours
of SQL practice.
"""

print(f"Blog post loaded: {len(blog_post)} characters")

## Step 2 — Convert to Twitter/X Thread

In [ ]:
thread_prompt = ChatPromptTemplate.from_template(
    """Convert this blog post into a Twitter/X thread (10-12 tweets).

BLOG POST:
{blog}

Rules:
- First tweet: Hook that makes people want to read more (no "Thread:" prefix)
- Each tweet: Max 280 characters
- Use short, punchy sentences
- Include relevant stats/numbers from the original
- Number each tweet (1/, 2/, etc.)
- Last tweet: Call to action or key takeaway
- Add appropriate line breaks between tweets
- Do NOT use hashtags (they reduce engagement)

Write the thread:"""
)

thread_chain = thread_prompt | llm | StrOutputParser()
thread = thread_chain.invoke({"blog": blog_post})
print("=== TWITTER THREAD ===")
print(thread)

## Step 3 — Convert to LinkedIn Post

In [ ]:
linkedin_prompt = ChatPromptTemplate.from_template(
    """Convert this blog post into a LinkedIn post.

BLOG POST:
{blog}

Rules:
- Opening line: Strong hook (this appears before "See more")
- Length: 200-300 words
- Tone: Professional but conversational
- Use short paragraphs (1-2 sentences each)
- Include 2-3 key stats or insights
- End with a question to drive engagement
- Use line breaks between paragraphs
- No emojis in every line (max 2-3 total if any)

Write the LinkedIn post:"""
)

linkedin_chain = linkedin_prompt | llm | StrOutputParser()
linkedin = linkedin_chain.invoke({"blog": blog_post})
print("=== LINKEDIN POST ===")
print(linkedin)

## Step 4 — Convert to Email Newsletter Blurb

In [ ]:
email_prompt = ChatPromptTemplate.from_template(
    """Convert this blog post into an email newsletter section.

BLOG POST:
{blog}

Rules:
- Subject line suggestion at the top
- Preview text (the snippet shown in inbox) — max 90 chars
- Body: 150-200 words
- Include 1 compelling stat
- End with a CTA link placeholder: [Read the full post →]
- Tone: Informative, concise, scannable

Write the newsletter blurb:"""
)

email_chain = email_prompt | llm | StrOutputParser()
email_blurb = email_chain.invoke({"blog": blog_post})
print("=== EMAIL NEWSLETTER ===")
print(email_blurb)

In [ ]:
# Full repurposing pipeline
def repurpose_blog(blog_text: str) -> dict:
    """Convert a blog post into multiple content formats."""
    results = {}
    formats = [
        ("twitter_thread", thread_chain),
        ("linkedin_post", linkedin_chain),
        ("email_blurb", email_chain),
    ]
    for name, chain in formats:
        print(f"Generating {name}...")
        results[name] = chain.invoke({"blog": blog_text})
    print("\nAll formats generated!")
    return results

all_formats = repurpose_blog(blog_post)
for fmt, content in all_formats.items():
    print(f"\n{'='*50}")
    print(f"{fmt.upper()}")
    print(f"{'='*50}")
    print(content)

## Summary & Next Steps

**What you learned:**
- ✅ One-input → multi-format content generation
- ✅ Format-specific prompt constraints
- ✅ Content repurposing pipelines
- ✅ Tone and length control through prompting

**Next steps:**
- Add quality scoring for each format
- Include hashtag suggestion for social posts
- Batch process multiple blog posts

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

prompt = PromptTemplate.from_template(
    "You are a helpful local AI assistant. Answer the user's question:\n\nQuestion: {question}\n\nAnswer:"
)

chain = prompt | llm | StrOutputParser()

# response = chain.invoke({"question": "What can you help me with?"})
# print(response)
print("LangChain inference pipeline ready!")
